<a href="https://colab.research.google.com/github/PetarGyurchev77/Revolut-Ombudsman-analysis/blob/main/Scrape_FOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import requests

url = "https://www.financial-ombudsman.org.uk/businesses/resolving-complaint/ombudsman-decisions/search"
params = {
    "BusinessName": "Revolut",
    "Business": "Revolut",
    "IndustrySectorID[1]": "1",
    "DateFrom": "2025-01-01",
    "DateTo": "2025-12-31",
    "Sort": "date",
    "Start": "0",
}
headers = {"User-Agent": "student-research-project/1.0 (personal study)"}

r = requests.get(url, params=params, headers=headers, timeout=30)
print(r.status_code, len(r.text))
print(r.url)

200 112068
https://www.financial-ombudsman.org.uk/businesses/resolving-complaint/ombudsman-decisions/search?BusinessName=Revolut&Business=Revolut&IndustrySectorID%5B1%5D=1&DateFrom=2025-01-01&DateTo=2025-12-31&Sort=date&Start=0


In [23]:
import re
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

node = soup.find(string=re.compile("DRN-"))
print(repr(node)[:200])

parent = node.find_parent(["li", "article", "div"])
print(parent.name, parent.get("class"))
print(parent.get_text(" ", strip=True)[:400])


'Decision Reference DRN-5697595'
li None
Decision Reference DRN-5697595 31 Dec 2025 Revolut Ltd Not upheld Banking and Payments DRN-5697595   The complaint  Ms K has complained about Revolut Ltd not refunding an authorised push payment she says she made and lost to a scam.  What happened The background to this complaint is well known to both parties, so I won’t repeat it in detail here. In summary, Ms K fell victim... (3 pages) View deci


In [24]:
import os, time
import pandas as pd

PRODUCTS = ["Banking and Payments", "Consumer Credit", "Insurance",
            "Investments", "Mortgages and Home Finance"]

HEAD = re.compile(
    r"Decision Reference (DRN-\d+)\s+(\d{1,2} \w{3} \d{4})\s+(.+?)\s+"
    r"(Not upheld|Upheld)\s+(.*)", re.S)

def parse_item(li):
    m = HEAD.search(li.get_text(" ", strip=True))
    if not m:
        return None
    ref, date, firm, outcome, rest = m.groups()
    rest = rest.split("View decision")[0].strip()
    rest = re.sub(r"\(\d+ pages?\)\s*$", "", rest).strip()
    product = next((p for p in sorted(PRODUCTS, key=len, reverse=True)
                    if rest.startswith(p)), None)
    if product:
        rest = rest[len(product):].strip()
    rest = re.sub(r"^DRN-\d+\s*", "", rest)      # repeated reference, when present
    return {"ref": ref, "date": date, "firm": firm,
            "outcome": outcome, "product": product, "summary": rest}

In [25]:
text = ("Decision Reference DRN-5247067 28 May 2025 Revolut Ltd Not upheld "
        "Banking and Payments The complaint Mrs A has complained about Revolut Ltd "
        "not refunding several payments she says she made and lost to a scam. "
        "What happened The background... (3 pages) View decision")

class Fake:                      # stand-in for a BeautifulSoup tag
    def get_text(self, *a, **k): return text

print(parse_item(Fake()))

{'ref': 'DRN-5247067', 'date': '28 May 2025', 'firm': 'Revolut Ltd', 'outcome': 'Not upheld', 'product': 'Banking and Payments', 'summary': 'The complaint Mrs A has complained about Revolut Ltd not refunding several payments she says she made and lost to a scam. What happened The background...'}


In [26]:
os.makedirs("pages", exist_ok=True)

def fetch_page(business, start):
    path = f"pages/{business}_{start}.html"
    if os.path.exists(path):                      # already downloaded: reuse
        return open(path, encoding="utf-8").read()
    p = dict(params, BusinessName=business, Business=business, Start=str(start))
    r = requests.get(url, params=p, headers=headers, timeout=30)
    r.raise_for_status()
    open(path, "w", encoding="utf-8").write(r.text)
    time.sleep(1)                                 # be polite
    return r.text

def parse_page(html):
    soup = BeautifulSoup(html, "html.parser")
    items = [li for li in soup.find_all("li")
             if li.find(string=re.compile(r"Decision Reference DRN-"))]
    return [parse_item(li) for li in items]

def scrape(business, max_pages=None):
    rows, start, pages = [], 0, 0
    while True:
        parsed = parse_page(fetch_page(business, start))
        if not parsed:
            break
        unparsed = sum(r is None for r in parsed)
        if unparsed:
            print(f"  {business} start={start}: {unparsed} rows failed to parse")
        rows += [dict(r, search_firm=business) for r in parsed if r]
        start += 10
        pages += 1
        if max_pages and pages >= max_pages:
            break
    print(f"{business}: {len(rows)} rows from {pages} pages")
    return rows

In [27]:

test = scrape("Revolut", max_pages=3)
len(test)          # expect 30


Revolut: 30 rows from 3 pages


30

In [28]:
all_rows = []
for business in ["Revolut", "Monzo", "Starling"]:
    all_rows += scrape(business)

df_raw = pd.DataFrame(all_rows).drop_duplicates("ref")
print(len(all_rows), "rows scraped;", len(df_raw), "after removing duplicate references")
df_raw.groupby(["search_firm", "outcome"]).size()


Revolut: 1727 rows from 173 pages
Monzo: 943 rows from 95 pages
Starling: 372 rows from 38 pages
3042 rows scraped; 3042 after removing duplicate references


search_firm  outcome   
Monzo        Not upheld     709
             Upheld         234
Revolut      Not upheld    1209
             Upheld         518
Starling     Not upheld     277
             Upheld          95
dtype: int64

In [29]:
def failures(business):
    out = []
    for f in sorted(os.listdir("pages")):
        if not f.startswith(business + "_"):
            continue
        soup = BeautifulSoup(open(f"pages/{f}", encoding="utf-8").read(), "html.parser")
        for li in soup.find_all("li"):
            if li.find(string=re.compile(r"Decision Reference DRN-")) and parse_item(li) is None:
                out.append((f, li.get_text(" ", strip=True)[:300]))
    return out

for f, text in failures("Revolut") + failures("Monzo") + failures("Starling"):
    print(f, "|", text, "\n")

In [30]:
df_raw.groupby(["search_firm", "firm"]).size()

search_firm  firm                           
Monzo        Monzo Bank Ltd                      943
Revolut      Revolut Ltd                        1727
Starling     Starling Bank Limited               267
             Startline Motor Finance Limited     105
dtype: int64

In [31]:
KEEP = df_raw["firm"].str.contains("Revolut|Monzo|Starling Bank", case=False)
print("dropping:", (~KEEP).sum())
print(df_raw.loc[~KEEP, "firm"].value_counts())

df = df_raw[KEEP].copy()
df.to_csv("fos_decisions_2025.csv", index=False)

print(len(df), "rows")
print(df.groupby(["search_firm", "outcome"]).size())
print("missing product:", df["product"].isna().sum())


dropping: 105
firm
Startline Motor Finance Limited    105
Name: count, dtype: int64
2937 rows
search_firm  outcome   
Monzo        Not upheld     709
             Upheld         234
Revolut      Not upheld    1209
             Upheld         518
Starling     Not upheld     214
             Upheld          53
dtype: int64
missing product: 0


In [32]:
RULES = [
    ("Gambling and vulnerability",
        r"(gambling|self-exclu).{0,80}(protect|block|stop|prevent|vulnerab)"
        r"|(protect|block|stop|prevent).{0,80}(gambling|self-exclu)"),
    ("Fraud (unauthorised)", r"unauthorised|didn'?t (make|authorise)|did not authorise"),
    ("Scams",               r"scam|fell victim|impersonat|blackmail|extortion"),
    ("Cifas marker",        r"cifas|fraud marker"),
    ("Blocked accounts/payments",
        r"block|restrict|froze|frozen|clos(e|ed|ing)\b.{0,30}\baccount|account (clos|review)"),
    ("Card disputes",
        r"chargeback|merchant|disputed transaction|debit card|cash machine|atm"
        r"|goods|not received|overcharg|refund"),
]

def classify(summary):
    text = str(summary).lower().replace("\u2019", "'")
    for area, pattern in RULES:
        if re.search(pattern, text):
            return area
    return "Other"

df["area"] = df["summary"].apply(classify)
df.to_csv("fos_classified_2025.csv", index=False)

print(pd.crosstab(df["area"], df["search_firm"]))
print()
print(df.groupby(["search_firm", "area"])["outcome"]
        .apply(lambda s: round(100 * (s == "Upheld").mean(), 1)).unstack(0))


search_firm                 Monzo  Revolut  Starling
area                                                
Blocked accounts/payments      87       39        27
Card disputes                  94       75        36
Cifas marker                  100       13        11
Fraud (unauthorised)           40       60         9
Gambling and vulnerability     11        8         4
Other                         121       62        39
Scams                         490     1470       141

search_firm                 Monzo  Revolut  Starling
area                                                
Blocked accounts/payments    16.1     33.3       7.4
Card disputes                30.9     26.7       2.8
Cifas marker                 14.0      7.7       9.1
Fraud (unauthorised)         10.0     23.3      22.2
Gambling and vulnerability   36.4      0.0       0.0
Other                        26.4     32.3      38.5
Scams                        28.0     30.6      22.7


In [33]:

check = df.sample(30, random_state=1)[["ref", "area", "summary"]]
for r in check.itertuples():
    print(r.area, "|", r.summary[:160], "\n")


Scams | The complaint  Mr and Mrs P has complained about Revolut Ltd not refunding several payments they say they made and lost to a scam.  What happened The background 

Scams | The complaint and background  Mr K complains Starling Bank Limited won’t reimburse around £24,000 that he lost when he fell victim to a cryptocurrency investmen 

Scams | The complaint  D, a limited company, complains that Starling Bank Limited won’t refund the money it lost when it was the victim of what it feels was a scam.  Wh 

Scams | The complaint  Mr A is unhappy that Monzo Bank Ltd failed to refund his lost funds after he fell victim to what he argues was an authorised push payment (APP) s 

Gambling and vulnerability | The complaint  Mr S complains Revolut Ltd failed to protect him from making payments to gambling operators. He says he suffered financially and mentally because 

Scams | The complaint  Mr R has complained that Monzo Bank Ltd won’t refund the money he says he lost in a scam. What happen

In [34]:
fresh = df.sample(30, random_state=7)[["ref", "area", "summary"]]
for r in fresh.itertuples():
    print(r.area, "|", r.summary[:160], "\n")

Scams | The complaint  Mrs H complains that Revolut Ltd hasn’t protected her from losing money to a scam.  What happened The background to this complaint is well known  

Fraud (unauthorised) | The complaint  Miss P complains Monzo Bank Ltd won’t refund her for payments made in connection with a scam – some of which she says were unauthorised. What hap 

Cifas marker | The complaint  Miss A has complained Monzo Bank Ltd lodged a fraud-related marker on the industry fraud database, CIFAS, in her name. What happened Miss A had h 

Scams | The complaint  Mr A complains that Monzo Bank Ltd won’t refund the money he lost when he was the victim of a scam.  What happened In late 2020, Mr A says he sta 

Scams | The complaint  Mr S complains Revolut Ltd won’t refund the money he lost to a scam. What happened In mid-2022, Mr S was researching investment opportunities whe 

Scams | The complaint  Mrs S complains Revolut Ltd won’t refund the money she lost when she fell victim to a scam. What hap

In [35]:

from google.colab import files
files.download("fos_classified_2025.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>